In [5]:
asset_1_returns = [
    2.1675, 3.3875, -2.5194, -1.3959, -2.7264, 0.7832, 3.9749, -3.3869, -0.1714, 0.5439,
    1.0913, -1.0361, -0.871, 1.0377, -2.7236, 2.7485, -0.8348, -1.2728, -0.5232, 0.1174,
    1.1843, 0.6003, -0.5785, -1.8834, 1.0666, 3.7147, -0.3213, 1.3712, -0.7987, 0.949,
    -2.5322, 0.4613, -1.6205, 4.7482, 1.6551, -5.1563, -1.0738, -1.2798, -0.7793, -0.3613,
    0.1896, -2.6047, -3.1645, -0.6066, 0.8595, 3.642, 0.6684, 1.5556, 1.4717, -2.6414,
    -1.6742, 1.5918, -1.4625, 0.0491, -0.2674, 1.0214, 1.0679, -5.4239, 0.4125, 4.9665

]

In [6]:
asset_2_returns = [
    -1.8883, -0.5608, 1.5773, 1.4225, 1.0408, 1.2914, -0.2969, 0.6602, 0.2207, 0.3632,
    0.103, -0.5964, 2.2177, 0.2439, 2.7658, -1.4885, -1.3056, 2.2504, 1.8409, 0.9127,
    1.6094, -0.1249, -0.2565, 0.5511, -0.555, 0.2948, 0.8969, -0.6688, -0.0603, -0.0582,
    3.4582, 1.0112, 0.7624, -2.1118, 0.6686, 1.286, 0.5736, -0.3968, -0.6159, -2.5366,
    0.889, 1.2735, 1.264, 1.6454, 0.1415, -0.3455, 1.9907, -1.1351, -1.3031, 0.9842,
    2.6791, -0.6697, -2.1408, 0.656, 0.0338, 1.0945, 0.9726, 3.2911, 0.2005, -2.4944
]

In [7]:
asset_3_returns = [
    3.0712, 1.2428, -3.184, 3.0923, -0.0131, -2.0597, 2.0496, -0.585, 0.6504, 1.0962,
    2.7193, -0.8361, -4.2316, 1.624, -3.2315, 7.4063, 0.3338, -4.3389, 1.5357, -1.2732,
    -0.208, -0.6808, 1.451, -1.5372, -1.0502, 3.36, -0.3788, -0.1197, -0.1823, -0.1217,
    -4.8028, -0.5589, -0.5277, 3.0093, 1.0955, -2.0927, 0.7881, -1.4555, 2.2117, 2.9634,
    -3.5207, -4.4262, -4.5345, 0.1385, -3.6547, 2.8741, -0.0495, 0.9447, 1.3656, 1.7807,
    -1.7593, 0.9009, -2.0389, 0.1216, 4.0254, 2.3588, 0.7781, -4.9794, 0.2146, -0.2641
]

In [8]:
mean_1 = sum(asset_1_returns) / len(asset_1_returns) * 12
mean_2 = sum(asset_2_returns) / len(asset_2_returns) * 12
mean_3 = sum(asset_3_returns) / len(asset_3_returns) * 12

print("Estimated annual mean for asset 1:", mean_1)
print("Estimated annual mean for asset 2:", mean_2)
print("Estimated annual mean for asset 3:", mean_3)

Estimated annual mean for asset 1: -0.5186400000000004
Estimated annual mean for asset 2: 4.7057400000000005
Estimated annual mean for asset 3: -0.6986200000000002


In [9]:
import numpy as np

cov_matrix = np.array([
    [0.0056, -0.0020, 0.0037],
    [-0.0020, 0.0022, -0.0022],
    [0.0037, -0.0022, 0.0074]
])

print("Estimated Covariance Matrix:\n", cov_matrix) 


Estimated Covariance Matrix:
 [[ 0.0056 -0.002   0.0037]
 [-0.002   0.0022 -0.0022]
 [ 0.0037 -0.0022  0.0074]]


In [12]:
volatility = 100*np.sqrt(np.diag(cov_matrix))
print("Non-annualized Volatility for each asset:", volatility)

Non-annualized Volatility for each asset: [7.48331477 4.69041576 8.60232527]


In [13]:
interest_rate = 1
excess_return = np.array([mean_1, mean_2, mean_3]) - interest_rate
print("Excess Return for each asset:", excess_return)

Excess Return for each asset: [-1.51864  3.70574 -1.69862]


In [42]:
from scipy.optimize import minimize

def optimize_portfolio_target_vol(cov_matrix, target_vol=0.05, bounds=None, max_iter=10000, tol=1e-9):
    """
    Find portfolio weights such that:
      - sum of weights == 1
      - portfolio volatility == target_vol
    Uses only the covariance matrix (no expected returns).
    If bounds are not provided, defaults to (0, 1) per asset.
    """
    Sigma = np.asarray(cov_matrix)
    n = Sigma.shape[0]

    # Default bounds: long-only [0, 1]
    if bounds is None:
        bounds = [(0.0, 1.0)] * n

    # Global minimum variance (GMV) portfolio as a good starting point
    ones = np.ones(n)
    try:
        A = np.linalg.solve(Sigma, ones)
    except np.linalg.LinAlgError:
        A = np.linalg.pinv(Sigma) @ ones
    w0 = A / A.sum()

    # Objective: pick the most diversified solution (minimize L2 norm) within the feasible set
    def obj(w):
        return 0.5 * np.dot(w, w)

    cons = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        {"type": "eq", "fun": lambda w: w @ Sigma @ w - target_vol ** 2},
    ]

    res = minimize(
        obj,
        w0,
        method="SLSQP",
        bounds=bounds,
        constraints=cons,
        options={"maxiter": max_iter, "ftol": tol, "disp": False},
    )

    w = res.x
    vol = float(np.sqrt(w @ Sigma @ w))
    return w, vol, res

# Use existing variables in the notebook if available
_target_vol = target_volatility if "target_volatility" in globals() else 0.05
_bounds = bounds if "bounds" in globals() else None

opt_weights_tv, opt_vol_tv, opt_res_tv = optimize_portfolio_target_vol(
    cov_matrix,
    target_vol=_target_vol,
    bounds=_bounds
)

print("Optimized weights (target vol):", opt_weights_tv)
print("Achieved volatility:", opt_vol_tv)
print("Sum of weights:", opt_weights_tv.sum())
print("Optimizer success:", opt_res_tv.success, "| Message:", opt_res_tv.message)

Optimized weights (target vol): [0.37642545 0.22074266 0.40283189]
Achieved volatility: 0.05000000000001873
Sum of weights: 1.0
Optimizer success: True | Message: Optimization terminated successfully


In [43]:
# Provided portfolio weights
weights = np.array([0.281380952897458, 0.238671354554862, 0.479947692547708])

# Estimated return using annualized means (with risk-free asset)
# Portfolio return = sum(weights * mean_i)
estimated_portfolio_return = np.dot(weights, [mean_1, mean_2, mean_3])
print("Estimated portfolio return (annualized):", estimated_portfolio_return)

# Realized (true) portfolio return using the actual data
# First, construct the portfolio returns for each period
portfolio_returns = (
    np.array(asset_1_returns) * weights[0] +
    np.array(asset_2_returns) * weights[1] +
    np.array(asset_3_returns) * weights[2]
)

# Realized mean return (annualized)
realized_portfolio_return = np.mean(portfolio_returns) * 12
print("Realized (true) portfolio return (annualized):", realized_portfolio_return)

Estimated portfolio return (annualized): 0.6418888656045787
Realized (true) portfolio return (annualized): 0.6418888656045788


In [44]:
import numpy as np

values = [
    -1.1168, -1.3565, 1.37536666666667, -1.03963333333333, 0.566233333333333,
    -0.0049666666666668, -1.9092, 1.1039, -0.233233333333333, -0.667766666666667,
    -1.30453333333333, 0.822866666666667, 0.961633333333333, -0.968533333333333,
    1.0631, -2.88876666666667, 0.6022, 1.12043333333333, -0.951133333333333,
    0.0810333333333334, -0.8619, 0.0684666666666667, -0.205333333333333,
    0.9565, 0.179533333333333, -2.4565, -0.0656, -0.194233333333333,
    0.3471, -0.256366666666667, 1.29226666666667, -0.304533333333333,
    0.461933333333333, -1.8819, -1.13973333333333, 1.98766666666667,
    -0.0959666666666666, 1.04403333333333, -0.272166666666667,
    -0.0218333333333334, 0.814033333333333, 1.91913333333333, 2.145,
    -0.392433333333333, 0.884566666666667, -2.05686666666667,
    -0.869866666666667, -0.455066666666667, -0.5114, -0.0411666666666667,
    0.251466666666667, -0.607666666666667, 1.88073333333333,
    -0.275566666666667, -1.26393333333333, -1.49156666666667,
    -0.939533333333333, 2.37073333333333, -0.275866666666667, -0.736
]


var_90 = np.percentile(values, 10)
var_90_rounded = round(var_90, 2)
print("90% VaR (rounded):", var_90_rounded)

90% VaR (rounded): -1.37


In [45]:
# CVaR (Conditional Value at Risk) at 90% level
cvar_90 = np.mean([v for v in values if v <= var_90])
cvar_90_rounded = round(cvar_90, 2)
print("90% CVaR (rounded):", cvar_90_rounded)

90% CVaR (rounded): -2.11


In [46]:
from scipy.stats import binom

# Parameters
n_years = 15
n_success = 12
p = 0.5

# Probability of 12 or more successes out of 15 (no skill, p=0.5)
prob = binom.sf(n_success - 1, n_years, p)
prob_rounded = round(prob, 4)
print("Probability of 12 or more successes out of 15 (p=0.5):", prob_rounded)

Probability of 12 or more successes out of 15 (p=0.5): 0.0176


In [47]:
from math import comb
p_tail = (comb(15,14) + comb(15,15)) / 2**15
prob = 1 - (1 - p_tail)**100
round(prob, 4)
# or with SciPy:
# from scipy.stats import binom
# prob = 1 - (1 - binom.sf(13, 15, 0.5))**100
# round(prob, 4)

0.0477

In [48]:
import random

def play(trials=1_00000):
    win = 0
    for _ in range(trials):
        doors = ['P','P','G','G']
        random.shuffle(doors)
        pick = random.randrange(4)
        # Monty opens a goat not equal to pick
        candidates = [i for i in range(4) if i != pick and doors[i]=='G']
        opened = random.choice(candidates)
        # always switch to a random unopened, not the original or opened
        choices = [i for i in range(4) if i not in (pick, opened)]
        new_pick = random.choice(choices)
        win += (doors[new_pick]=='P')
    return win / trials

print(round(play(), 4))  # ~0.75

0.7509
